# IR pricing and risk on the toy assets: swaps, swaptions, bonds

A tour of the interest-rate surface pricebt ports from gs_quant (`docs/v2/IR_RISK_DESIGN.md`),
on the deterministic toy assets under `tests/assets/` (a flat-curve rates world, a Bachelier
swaption, a toy bond master), so it runs anywhere with no market data:

1. the **measure contract** an `IRSwaption` config must satisfy;
2. pricing a Portfolio of an `IRSwaption`, a `Bond` and an `IRSwap` with several measures: scalars,
   a bucketed delta ladder, a vega cube and the `Cashflows` table;
3. `Portfolio` paths, `subset`, `to_frame` and `aggregate`, and a historical calc;
4. measure parameters (`bump_size`, ...);
5. a delta-hedged swaption backtest explained by `swaption_pnl_definition()` and
   `pnl_explain_table()`, a bond backtest whose coupon shows up as cash, and
   `PnlExplain(CloseMarket(...))` between two dates.

Only the imports and `PricebtSession.use(...)` differ from gs code.

In [1]:
import sys
from datetime import date

import pandas as pd

for _p in ("src", "tests"):  # the toy library lives under tests/ (run from the repo root)
    if _p not in sys.path:
        sys.path.insert(0, _p)

from pricebt.backtests.actions import AddTradeAction, HedgeAction
from pricebt.backtests.backtest_objects import bond_pnl_definition, swaption_pnl_definition
from pricebt.backtests.generic_engine import GenericEngine
from pricebt.backtests.strategy import Strategy
from pricebt.backtests.triggers import DateTrigger, DateTriggerRequirements, PeriodicTrigger, PeriodicTriggerRequirements
from pricebt.errors import NotSupportedError
from pricebt.instrument import Bond, IRSwap, IRSwaption
from pricebt.markets import CloseMarket, HistoricalPricingContext, PricingContext
from pricebt.markets.portfolio import Portfolio
from pricebt.risk import (Cashflows, IRDelta, IRDeltaParallel, IRFwdRate, IRGammaParallel, IRVega, IRVegaParallel,
                          PnlExplain, Price, Theta, contracts)
from pricebt.session import PricebtSession

ASSETS = ["tests/assets/toy_usd_irs_full.yaml", "tests/assets/toy_usd_swaption.yaml", "tests/assets/toy_usd_bond.yaml"]
PricebtSession.use(assets=ASSETS)    # replaces GsSession.use(...)
D = date(2024, 3, 4)

## 1. The measure contract

Every `IRSwap`, `IRSwaption` and `Bond` config must map each measure (and form) of its class's
contract to a function with an allowed unit, or declare it under `unsupported_measures:` with a
reason; loading fails otherwise, listing every gap (DEV-I11). The contract is data in
`pricebt.risk.contracts`:

In [2]:
contract = pd.DataFrame(
    [(r.measure, r.kind, ", ".join(r.forms), ", ".join(sorted(contracts.KINDS[r.kind][0] or ["frame"])))
     for r in contracts.contract_for("IRSwaption")],
    columns=["measure", "kind", "forms", "allowed units"],
)
contract

,measure,kind,forms,allowed units
0,Price,value,scalar,ccy
1,IRDelta,sens1,"scalar, bucketed",ccy_per_bp
2,IRDiscountDeltaParallel,sens1,scalar,ccy_per_bp
3,IRGammaParallel,sens2,scalar,ccy_per_bp2
4,IRGamma,sens2,bucketed,ccy_per_bp2
5,IRVega,sens1,"scalar, bucketed",ccy_per_bp
6,IRVanna,sens2,scalar,ccy_per_bp2
7,IRVolga,sens2,scalar,ccy_per_bp2
8,IRBasis,sens1,scalar,ccy_per_bp
9,IRXccyDelta,sens1,scalar,ccy_per_bp


A library that cannot compute a measure pastes the block the load error prints, then writes the
real reason in place of each `TODO`:

In [3]:
print(contracts.unsupported_block("IRSwaption", [("IRVanna", "scalar"), ("IRVega", "bucketed")]))

unsupported_measures:
  IRVega: {bucketed: "TODO: why your library cannot compute this"}
  IRVanna: "TODO: why your library cannot compute this"



## 2. Pricing a Portfolio

A bought 1y10y ATM payer swaption, a long toy bond and a 10y pay-fixed swap, the two linear
trades in a sub-portfolio. Each instrument is priced by the asset config that matches it.

In [4]:
swaption = IRSwaption("Pay", "10y", "USD", notional_amount=1e6, expiration_date="1y", strike="ATM", buy_sell="Buy", name="swaption")
bond = Bond(identifier="TOY 4.25 2034-11-15", size=1e6, buy_sell="Buy", settlement_currency="USD", name="bond")
swap = IRSwap("Pay", "10y", "USD", 1e6, fixed_rate="ATM", name="swap")
book = Portfolio((swaption, Portfolio((bond, swap), name="linear")), name="book")

with PricingContext(D):
    book.resolve()
    scalars = book.calc((Price, IRDeltaParallel, IRGammaParallel, IRVegaParallel, Theta, IRFwdRate))

scalars.to_frame()

risk_measure                             Price  \
portfolio_name_0 instrument_name                 
N/A              swaption         3.059590e+04   
linear           bond             1.008764e+06   
                 swap             0.000000e+00   

risk_measure                      IRDeltaParallel(aggregation_level:Asset)  \
portfolio_name_0 instrument_name                                             
N/A              swaption                                       372.186240   
linear           bond                                          -867.448729   
                 swap                                           811.888271   

risk_measure                      IRGammaParallel  \
portfolio_name_0 instrument_name                    
N/A              swaption                2.721644   
linear           bond                    0.859215   
                 swap                   -0.808956   

risk_measure                      IRVegaParallel(aggregation_level:Asset)  \
portfolio_name_0 instrument_name                                            
N/A              swaption                                      311.471727   
linear           bond                                            0.000000   
                 swap                                            0.000000   

risk_measure                             Theta   IRFwdRate  
portfolio_name_0 instrument_name                            
N/A              swaption        -3.866647e+01  398.905633  
linear           bond             1.174192e+02  424.831831  
                 swap             5.551115e-11  398.905465

Units: `Price` and `Theta` in USD (Theta per calendar day), deltas per +1bp of each instrument's
own rate (`IRFwdRate`: swap par rate, swaption forward, bond yield, all in bp), gamma per bp², vega
per bp of normal vol. The bond's delta is negative (long bond, per +1bp of yield).

Bucketed forms: the book's delta ladder (summed across instruments by pillar), the swaption's
vega cube keyed `'<tail>;<expiry>'`, and the bond's `Cashflows` table.

In [5]:
with PricingContext(D):
    ladder = book.calc(IRDelta).aggregate()
    cube = swaption.calc(IRVega).result()
    flows = bond.calc(Cashflows).result()
ladder

,mkt_type,mkt_asset,mkt_class,mkt_point,mkt_quoting_style,value
0,IR,TOY-USD,OIS,2Y,,-22.945987
1,IR,TOY-USD,OIS,5Y,,-72.853039
2,IR,TOY-USD,OIS,10Y,,412.424808
3,IR,TOY-USD,OIS,30Y,,0.000000


In [6]:
cube[cube.value.abs() > 1.0]

,mkt_type,mkt_asset,mkt_class,mkt_point,mkt_quoting_style,value
0,IR VOL,TOY-USD,SWAPTION,10Y;1Y,,311.471727


In [7]:
flows.head()

,payment_date,payment_amount,currency,payment_type,accrual_start_date,accrual_end_date,notional,rate
0,2024-05-15,21250.0,USD,Coupon,2023-11-15,2024-05-15,1000000.0,0.0425
1,2024-11-15,21250.0,USD,Coupon,2024-05-15,2024-11-15,1000000.0,0.0425
2,2025-05-15,21250.0,USD,Coupon,2024-11-15,2025-05-15,1000000.0,0.0425
3,2025-11-15,21250.0,USD,Coupon,2025-05-15,2025-11-15,1000000.0,0.0425
4,2026-05-15,21250.0,USD,Coupon,2025-11-15,2026-05-15,1000000.0,0.0425


## 3. Portfolio paths, subset, to_frame, aggregate, history

In [8]:
print([str(p) for p in book.all_paths])
bond_path = book.paths("bond")
print(bond_path, book[bond_path[0]].name)

linear = scalars.subset(book.paths("swap") + bond_path)
linear.to_frame()

['(0,)', '(1, 0)', '(1, 1)']
((1, 0),) bond


risk_measure,Price,IRDeltaParallel(aggregation_level:Asset),IRGammaParallel,IRVegaParallel(aggregation_level:Asset),Theta,IRFwdRate
instrument_name,,,,,,
swap,0.000000e+00,811.888271,-0.808956,0.0,5.551115e-11,398.905465
bond,1.008764e+06,-867.448729,0.859215,0.0,1.174192e+02,424.831831


In [9]:
book_delta = scalars[IRDeltaParallel].aggregate()
hedge_ratio = -book_delta / float(scalars[swap][IRDeltaParallel])  # swaps that flatten the book
print(f"book IRDeltaParallel {float(book_delta):,.2f} USD/bp; hedge with {hedge_ratio:.3f} x the swap")

book IRDeltaParallel 316.63 USD/bp; hedge with -0.390 x the swap


In [10]:
with HistoricalPricingContext(date(2024, 3, 4), date(2024, 3, 8)):
    history = book.calc(Price)
history.aggregate()

2024-03-04    1.039360e+06
2024-03-05    1.039246e+06
2024-03-06    1.039115e+06
2024-03-07    1.038968e+06
2024-03-08    1.038804e+06
dtype: float64

## 4. Measure parameters

`bump_size`, `finite_difference_method`, `local_curve` and `scale_factor` reach a config function
as `pricebt_bump_size`, ... (DEV-I10). None of the toy functions takes a bump size, so pricebt
refuses the request instead of silently ignoring it:

In [11]:
try:
    with PricingContext(D):
        swaption.calc(IRDelta(aggregation_level="Type", bump_size=5))
except NotSupportedError as exc:
    print(exc)

asset toy_usd_swaption: IRDelta(aggregation_level:Type, bump_size:5) sets bump_size; function 'delta' does not reference pricebt_bump_size


## 5. P&L decomposition

### A delta-hedged swaption

The swaption bought on the first day, delta-hedged daily with a 10y payer held for one business
day. `swaption_pnl_definition()` attributes each step to delta, gamma, vega, vanna, volga and theta;
`pnl_explain_table()` puts that next to the actual P&L. The hedges add delta, gamma and theta but no
vol terms (a swap's vega is 0).

In [12]:
start, end = date(2024, 1, 2), date(2024, 1, 31)
hedge = HedgeAction(IRDeltaParallel, IRSwap("Pay", "10y", "USD", 1e6, fixed_rate="ATM", name="hedge"), "1b", name="Hedge")
triggers = [
    DateTrigger(DateTriggerRequirements(dates=[start]), [AddTradeAction(IRSwaption("Pay", "10y", "USD", notional_amount=1e6, expiration_date="1y", buy_sell="Buy", name="swaption"), name="Add")]),
    PeriodicTrigger(PeriodicTriggerRequirements(start_date=start, end_date=end, frequency="1b"), [hedge]),
]
hedged = GenericEngine().run_backtest(Strategy(None, triggers), start=start, end=end, frequency="1b",
                                      pnl_explain=swaption_pnl_definition(), show_progress=False)
table = hedged.pnl_explain_table()
table.round(2).head()

,actual_pnl,cashflow_pnl,economic_pnl,PNL_delta,PNL_gamma,VegaPnL,PNL_vanna,PNL_volga,PNL_theta,explained_pnl,residual_pnl
2024-01-03,62.08,0.0,62.08,-0.01,4.04,96.21,-0.09,0.0,-37.99,62.16,-0.08
2024-01-04,58.40,0.0,58.40,-0.01,3.77,92.87,-0.11,0.0,-38.06,58.47,-0.06
2024-01-05,54.69,0.0,54.69,-0.01,3.51,89.48,-0.12,0.0,-38.11,54.74,-0.05
2024-01-08,-25.63,0.0,-25.63,-0.01,3.25,86.02,-0.13,0.0,-114.47,-25.34,-0.29
2024-01-09,46.82,0.0,46.82,-0.01,3.01,82.31,-0.14,0.0,-38.32,46.85,-0.03


In [13]:
residual_summary = pd.Series({
    "steps": len(table),
    "sum |actual_pnl|": table["actual_pnl"].abs().sum(),
    "sum |residual_pnl|": table["residual_pnl"].abs().sum(),
    "max |residual_pnl|": table["residual_pnl"].abs().max(),
    "residual / actual": table["residual_pnl"].abs().sum() / table["actual_pnl"].abs().sum(),
})
residual_summary.round(4)

steps                  21.0000
sum |actual_pnl|      689.3591
sum |residual_pnl|      1.2017
max |residual_pnl|      0.2917
residual / actual       0.0017
dtype: float64

### A bond across a coupon date

`Price` drops each coupon on its payment date and `Cashflows` lists the flows still to drop, so
`cashflow_pnl` carries the coupon (face 1mm x 4.25% / 2 = 21,250) on the step that pays it, and the
economic P&L stays smooth.

In [14]:
bond_bt = GenericEngine().run_backtest(
    Strategy(None, DateTrigger(DateTriggerRequirements(dates=[date(2024, 5, 6)]), [AddTradeAction(Bond(identifier="TOY 4.25 2034-11-15", size=1e6, buy_sell="Buy", settlement_currency="USD", name="bond"), name="Add")])),
    start=date(2024, 5, 6), end=date(2024, 5, 20), frequency="1b", risks=[Cashflows],
    pnl_explain=bond_pnl_definition(), show_progress=False)
bond_table = bond_bt.pnl_explain_table()
bond_table[["actual_pnl", "cashflow_pnl", "economic_pnl", "explained_pnl", "residual_pnl"]].round(2)

,actual_pnl,cashflow_pnl,economic_pnl,explained_pnl,residual_pnl
2024-05-07,2604.26,0.0,2604.26,2604.84,-0.58
2024-05-08,2601.40,0.0,2601.40,2601.98,-0.58
2024-05-09,2596.87,0.0,2596.87,2597.45,-0.58
2024-05-10,2590.68,0.0,2590.68,2591.26,-0.58
2024-05-13,2773.53,0.0,2773.53,2775.26,-1.73
2024-05-14,2572.11,0.0,2572.11,2572.69,-0.58
2024-05-15,-18689.12,21250.0,2560.88,2561.46,-0.58
2024-05-16,2546.21,0.0,2546.21,2546.78,-0.57
2024-05-17,2531.66,0.0,2531.66,2532.23,-0.56
2024-05-20,2696.97,0.0,2696.97,2698.64,-1.67


### `PnlExplain` between two dates

gs's instrument-level explain (notebook 030007): full revaluation of the book from the market of
one date to the market of another, by risk factor. The time component is not part of it.

In [15]:
F, T = date(2024, 3, 4), date(2024, 3, 11)
explain = PnlExplain(CloseMarket(date=T))
with PricingContext(pricing_date=F):
    explained = book.calc((Price, explain))
with PricingContext(pricing_date=F, market=CloseMarket(date=T)):
    moved = book.calc(Price)
explain_rows = explained[explain].aggregate()
print(f"market move at fixed time: {float(moved.aggregate()) - float(explained[Price].aggregate()):,.2f} USD")
explain_rows

market move at fixed time: -290.03 USD


,mkt_type,mkt_asset,mkt_class,mkt_point,mkt_quoting_style,value
0,IR,USD,,,,-1234.148326
1,IR VOL,USD,,,,-362.646349
2,CROSSES,,,,,4.782897
3,CREDIT,USD,,,,1301.977631
